# LangGraph: 멀티 에이전트와 Structured Output

이전 노트북(04-2)에서 Tool Use와 ReAct 루프를 학습했습니다. 이번에는 **여러 에이전트가 협업하는 멀티 에이전트 패턴**과 **구조화된 출력(Structured Output)**을 구현합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Structured Output | Pydantic 모델로 LLM 출력 형식을 강제 |
| 멀티 에이전트 | Worker + Evaluator 패턴 |
| 복합 State | messages 외에 여러 필드를 가진 상태 |
| 평가 기반 루프 | Evaluator가 품질을 판단하여 재작업 또는 완료 |
| 조건부 라우팅 | 여러 조건에 따라 다른 노드로 분기 |

## 학습 목표

1. `with_structured_output`으로 LLM 출력을 **Pydantic 모델**로 받기
2. **Worker-Evaluator 패턴**으로 자동 품질 검증 루프 구현하기
3. messages 외에 여러 필드를 가진 **복합 State** 설계하기
4. 여러 조건부 Edge로 **복잡한 분기 로직** 구현하기
5. 멀티 에이전트 시스템을 **Gradio UI**로 실행하기

---

## 이전 노트북과의 비교

| | 04-1 (Basic) | 04-2 (Tool Use) | 04-3 (Multi-Agent) |
|---|---|---|---|
| **노드 수** | 1개 | 2개 | **3개** (worker, tools, evaluator) |
| **에이전트 수** | 1개 | 1개 | **2개** (worker LLM, evaluator LLM) |
| **State** | messages만 | messages만 | **복합** (messages + 여러 필드) |
| **루프** | 없음 | chatbot↔tools | worker↔tools + **worker↔evaluator** |
| **출력 형식** | 자유 텍스트 | 자유 텍스트 | **Pydantic 구조화** |

---

## 1. Structured Output (구조화된 출력)

### 문제: LLM의 출력은 자유 텍스트

LLM에게 "이 응답이 좋은지 평가해줘"라고 요청하면, 답변 형식이 매번 달라질 수 있습니다:

```
응답 A: "네, 잘 했습니다. 성공 기준을 충족합니다."
응답 B: "부족합니다. 더 구체적인 예시가 필요합니다. 사용자에게 질문이 필요합니다."
응답 C: "Overall good, but criteria not fully met."
```

이렇게 자유 텍스트로 받으면 **프로그래밍으로 처리하기 어렵습니다**. "성공했는지 여부"를 코드에서 판단하려면 텍스트를 파싱해야 합니다.

### 해결: Pydantic 모델로 출력 강제

`with_structured_output`을 사용하면 LLM이 **반드시 정해진 형식**으로 응답합니다:

```
┌─────────────────────────────────────────────────────────────────────┐
│  자유 텍스트 출력 vs Structured Output                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  일반 LLM:                                                         │
│    llm.invoke("평가해줘")                                          │
│    → "네, 잘 했습니다. 성공 기준을 충족합니다."  (str)             │
│       ↑ 파싱이 어려움                                              │
│                                                                     │
│  Structured Output:                                                │
│    llm.with_structured_output(EvaluatorOutput).invoke("평가해줘")  │
│    → EvaluatorOutput(                                              │
│        feedback="잘 했습니다",                                     │
│        success_criteria_met=True,     ← bool로 바로 사용 가능!     │
│        user_input_needed=False         ← bool로 바로 사용 가능!    │
│      )                                                             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### bind_tools vs with_structured_output

| | `bind_tools` | `with_structured_output` |
|---|---|---|
| **용도** | 도구 호출 (함수 실행) | 출력 형식 강제 |
| **반환** | AIMessage (tool_calls 포함) | **Pydantic 객체** 직접 반환 |
| **사용 위치** | 도구를 쓰는 Worker | 평가/분류/추출 등 |
| **예시** | 날씨 API 호출 | 합격/불합격 판정 |

---

## 2. Worker-Evaluator 멀티 에이전트 패턴

### 아이디어

사람이 일할 때도 **작업자(Worker)**와 **검수자(Evaluator)**가 따로 있습니다. 같은 패턴을 LLM 에이전트에 적용합니다:

```
┌─────────────────────────────────────────────────────────────────────┐
│              Worker-Evaluator 패턴                                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   사용자: "CNN 뉴스를 요약해줘" (성공 기준: 3줄 이내)             │
│       │                                                             │
│       ▼                                                             │
│   ┌──────────┐                                                     │
│   │  Worker   │  도구를 사용해서 작업 수행                         │
│   │  (LLM 1)  │  (브라우저로 CNN 방문 → 텍스트 추출 → 요약)       │
│   └──────────┘                                                     │
│       │                                                             │
│       ▼                                                             │
│   ┌──────────┐                                                     │
│   │Evaluator │  성공 기준 충족 여부 판단                           │
│   │ (LLM 2)  │  → Structured Output으로 결과 반환                  │
│   └──────────┘                                                     │
│       │                                                             │
│   ┌───┴───┐                                                       │
│   │       │                                                       │
│   ▼       ▼                                                       │
│  통과    실패 → 피드백과 함께 Worker에게 다시 보냄                 │
│  (END)        → "3줄이 아니라 5줄입니다. 더 압축하세요"           │
│               → Worker가 피드백 반영하여 재작업                    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 왜 에이전트를 분리하는가?

| 하나의 LLM으로 | Worker + Evaluator 분리 |
|---|---|
| "잘 했는지 스스로 판단해" → 항상 "잘 했다"고 함 | 별도 LLM이 객관적으로 평가 |
| 자기 실수를 못 발견 | 피드백 루프로 품질 향상 |
| 한 번에 끝 (재시도 없음) | 기준 미달이면 자동 재작업 |

### 3가지 종료 조건

Evaluator는 Structured Output으로 3가지를 판단합니다:

1. **`success_criteria_met = True`** → 작업 완료, END
2. **`user_input_needed = True`** → 사용자에게 질문 필요, END (사용자 입력 대기)
3. **둘 다 False** → 재작업 필요, Worker에게 피드백과 함께 되돌려 보냄

---

## 3. 환경 설정

In [ ]:
from typing import Annotated, List, Dict, Any, Optional
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv

load_dotenv(override=True)

import os
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

---

## 4. Structured Output 정의

Evaluator가 반환할 출력 형식을 **Pydantic 모델**로 정의합니다. `Field(description=...)`은 LLM에게 각 필드의 의미를 알려주는 역할을 합니다.

In [ ]:
class EvaluatorOutput(BaseModel):
    """Evaluator가 반환하는 구조화된 평가 결과"""
    feedback: str = Field(
        description="Worker의 응답에 대한 구체적인 피드백"
    )
    success_criteria_met: bool = Field(
        description="성공 기준이 충족되었는지 여부"
    )
    user_input_needed: bool = Field(
        description="사용자의 추가 입력이 필요한지 여부 (질문, 명확화 요청, 또는 Worker가 막혔을 때)"
    )

# Structured Output 동작 확인
test_llm = ChatOpenAI(model="gpt-4o-mini")
test_llm_structured = test_llm.with_structured_output(EvaluatorOutput)

result = test_llm_structured.invoke(
    "다음 응답을 평가해줘. 성공 기준: '3줄 이내 요약'. 응답: '오늘 날씨가 좋습니다. 산책하기 좋은 날입니다.'"
)

print(f"타입: {type(result).__name__}")
print(f"feedback: {result.feedback}")
print(f"success_criteria_met: {result.success_criteria_met}")
print(f"user_input_needed: {result.user_input_needed}")

반환값이 **str이 아니라 Pydantic 객체**입니다. `result.success_criteria_met`처럼 필드에 직접 접근할 수 있어서, 조건부 분기에서 바로 활용할 수 있습니다.

---

## 5. 복합 State 설계

지금까지는 `messages`만 가진 단순한 State를 사용했습니다. 멀티 에이전트에서는 **더 많은 정보를 State에 저장**해야 합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  04-1, 04-2의 단순 State        04-3의 복합 State                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  class State:                   class State:                       │
│    messages ← reducer 적용        messages ← reducer 적용          │
│                                   success_criteria ← 덮어쓰기     │
│                                   feedback_on_work ← 덮어쓰기     │
│                                   success_criteria_met ← 덮어쓰기 │
│                                   user_input_needed ← 덮어쓰기    │
│                                                                     │
│  1개 필드                       5개 필드                           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Reducer가 있는 필드 vs 없는 필드**:
- `messages`: `add_messages` reducer → 새 값이 **이어붙여짐** (대화 누적)
- 나머지 필드: reducer 없음 → 새 값이 **덮어쓰기됨** (최신 값만 유지)

| 필드 | 타입 | 역할 | 업데이트 방식 |
|------|------|------|------|
| `messages` | list (reducer) | 대화 기록 | 이어붙이기 |
| `success_criteria` | str | 성공 기준 (사용자가 설정) | 덮어쓰기 |
| `feedback_on_work` | str / None | Evaluator의 피드백 | 덮어쓰기 |
| `success_criteria_met` | bool | 성공 기준 충족 여부 | 덮어쓰기 |
| `user_input_needed` | bool | 사용자 입력 필요 여부 | 덮어쓰기 |

In [ ]:
# 복합 State 정의 — TypedDict 사용

class State(TypedDict):
    messages: Annotated[List[Any], add_messages]   # reducer: 이어붙이기
    success_criteria: str                           # 덮어쓰기
    feedback_on_work: Optional[str]                 # 덮어쓰기
    success_criteria_met: bool                      # 덮어쓰기
    user_input_needed: bool                         # 덮어쓰기

print("State 정의 완료")
print("필드:", list(State.__annotations__.keys()))

### TypedDict vs BaseModel

이번에는 `BaseModel` 대신 `TypedDict`를 사용합니다. 둘 다 State로 사용할 수 있습니다:

| | TypedDict | BaseModel (Pydantic) |
|---|---|---|
| **접근** | `state["messages"]` (dict) | `state.messages` (attribute) |
| **검증** | 런타임 검증 없음 | 타입 검증 자동 |
| **사용** | 가볍고 간단 | 엄격한 검증 필요 시 |

---

## 6. 도구 설정 — Playwright 브라우저

LangChain 커뮤니티의 **Playwright 브라우저 도구킷**을 사용합니다. Worker가 실제 웹 브라우저를 조작하여 웹사이트를 방문하고 텍스트를 추출할 수 있습니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  Playwright 브라우저 도구킷                                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  navigate_browser   →  URL로 이동                                  │
│  extract_text       →  페이지 텍스트 추출                          │
│  click_element      →  버튼/링크 클릭                              │
│  fill_text          →  입력 필드에 텍스트 입력                     │
│  get_elements       →  HTML 요소 검색                              │
│  current_page       →  현재 URL 확인                               │
│                                                                     │
│  → LLM이 이 도구들을 조합하여 웹사이트를 탐색합니다               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 사전 준비

```bash
pip install playwright langchain-community
playwright install
```

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser

# headless=False: 실제 브라우저 창이 열림 (동작 확인 가능)
# headless=True: 백그라운드 실행 (서버 환경)
async_browser = create_async_playwright_browser(headless=False)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

# 사용 가능한 브라우저 도구 목록
print("=== Playwright 브라우저 도구 목록 ===")
for tool in tools:
    print(f"  {tool.name}: {tool.description[:60]}...")

In [ ]:
# 브라우저 도구 직접 테스트 — 웹사이트 방문 + 텍스트 추출
import textwrap

tool_dict = {tool.name: tool for tool in tools}
navigate_tool = tool_dict["navigate_browser"]
extract_text_tool = tool_dict["extract_text"]

# 웹사이트 방문
await navigate_tool.arun({"url": "https://www.example.com"})

# 텍스트 추출
text = await extract_text_tool.arun({})
print(textwrap.fill(text, width=80))

### LLM 초기화

**두 개의 LLM**을 사용합니다 — Worker용과 Evaluator용. 같은 모델이지만 **역할이 다릅니다**.

- **Worker LLM**: `bind_tools`로 Playwright 브라우저 도구를 연결
- **Evaluator LLM**: `with_structured_output`으로 EvaluatorOutput 형식 강제

In [ ]:
# Worker LLM — Playwright 도구 사용 가능
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

# Evaluator LLM — Structured Output 사용
evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

print("Worker LLM: bind_tools (Playwright 브라우저 도구 연결)")
print("Evaluator LLM: with_structured_output (구조화된 평가)")

---

## 7. 노드 구현

### 전체 그래프 구조 미리보기

```
┌─────────────────────────────────────────────────────────────────────┐
│                멀티 에이전트 그래프 구조                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│         START                                                      │
│           │                                                        │
│           ▼                                                        │
│     ┌──────────┐                                                   │
│     │  worker   │ ◄─────────────────────┐                         │
│     │  (LLM 1)  │                       │                         │
│     └──────────┘                        │                         │
│      │         │                        │                         │
│  tool_calls  tool_calls                │                         │
│  있음 ✓      없음 ✗                    │                         │
│      │         │                        │                         │
│      ▼         ▼                        │                         │
│  ┌────────┐  ┌───────────┐              │                         │
│  │ tools  │  │ evaluator │              │  피드백 + 재작업 요청   │
│  │        │  │  (LLM 2)  │              │                         │
│  └────────┘  └───────────┘              │                         │
│      │        │         │               │                         │
│      │    성공 또는    기준 미달         │                         │
│      │    사용자 입력   ─────────────────┘                         │
│      │    필요                                                     │
│      │        │                                                    │
│      └──► worker       END                                        │
│           로 복귀       (완료)                                     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**두 개의 루프**가 있습니다:
1. **Worker ↔ Tools**: 도구 사용 루프 (04-2와 동일)
2. **Worker ↔ Evaluator**: 품질 검증 루프 (이번 노트북의 핵심)

### 7-1. Worker 노드

Worker는 사용자의 요청을 수행하는 **작업자 에이전트**입니다.

- 도구를 사용할 수 있습니다 (bind_tools)
- **System Message**에 성공 기준과 이전 피드백을 포함합니다
- 이전에 Evaluator가 거부한 경우, 피드백을 반영하여 재작업합니다

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    """작업을 수행하는 Worker 노드"""
    
    # System Message 구성 — 성공 기준을 포함
    system_message = f"""You are a helpful assistant that can use tools to complete tasks.
You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
This is the success criteria:
{state['success_criteria']}
You should reply either with a question for the user about this assignment, or with your final response.
If you have a question for the user, you need to reply by clearly stating your question. An example might be:

Question: please clarify whether you want a summary or a detailed answer

If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.
"""
    
    # Evaluator가 이전에 거부한 경우 → 피드백을 System Message에 추가
    if state.get("feedback_on_work"):
        system_message += f"""
Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state['feedback_on_work']}
With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""
    
    # System Message 업데이트 또는 추가
    messages = state["messages"]
    found_system_message = False
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages
    
    # LLM 호출 (도구 사용 가능)
    response = worker_llm_with_tools.invoke(messages)
    
    return {"messages": [response]}

print("Worker 노드 정의 완료")

### 7-2. Worker 라우터

Worker의 응답을 보고 **다음 노드를 결정**합니다:
- `tool_calls`가 있으면 → `"tools"` (도구 실행)
- `tool_calls`가 없으면 → `"evaluator"` (평가 요청)

04-2의 `tools_condition`과 비슷하지만, END 대신 **evaluator로 보내는** 것이 차이점입니다.

In [ ]:
def worker_router(state: State) -> str:
    """Worker의 응답에 따라 다음 노드를 결정"""
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"      # 도구 실행 필요
    else:
        return "evaluator"  # 작업 완료 → 평가 요청

print("Worker 라우터 정의 완료")
print("  tool_calls 있음 → tools")
print("  tool_calls 없음 → evaluator")

### 7-3. Evaluator 노드

Evaluator는 Worker의 결과를 **성공 기준에 따라 평가**하는 검수자입니다.

핵심 특징:
- `with_structured_output`으로 **EvaluatorOutput** 형식의 결과를 받음
- Worker와 **별도의 대화 컨텍스트**를 사용 (Worker의 messages를 요약해서 전달)
- 평가 결과를 State에 저장하여 라우터가 분기에 활용

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    """대화 기록을 텍스트로 포매팅"""
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation


def evaluator(state: State) -> State:
    """Worker의 결과를 평가하는 Evaluator 노드"""
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""
    
    user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.
"""
    # 이전 피드백이 있으면 추가 (반복 실수 방지)
    if state["feedback_on_work"]:
        user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
        user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."
    
    # Evaluator는 별도의 messages로 호출 (Worker의 대화와 분리)
    evaluator_messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=user_message)
    ]

    # Structured Output으로 평가 결과 받기
    eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
    
    # 평가 결과를 State에 반영
    new_state = {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback: {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed
    }
    return new_state

print("Evaluator 노드 정의 완료")

### Evaluator의 핵심 패턴

```
┌─────────────────────────────────────────────────────────────────────┐
│  Evaluator가 하는 일                                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  입력:                                                             │
│    - Worker의 전체 대화 기록 (format_conversation)                  │
│    - 성공 기준 (state['success_criteria'])                          │
│    - Worker의 마지막 응답                                          │
│    - 이전 피드백 (있으면)                                          │
│                                                                     │
│  처리:                                                             │
│    evaluator_llm_with_output.invoke(...)                           │
│    → EvaluatorOutput 객체로 반환                                   │
│                                                                     │
│  출력 (State 업데이트):                                            │
│    feedback_on_work = "구체적 피드백 내용"                         │
│    success_criteria_met = True / False                             │
│    user_input_needed = True / False                                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**중요**: Evaluator는 Worker의 `messages`를 직접 사용하지 않고, **별도의 messages로 호출**합니다. Worker의 대화 기록은 텍스트로 요약해서 전달합니다. 이렇게 분리하면 두 LLM의 맥락이 섞이지 않습니다.

### 7-4. Evaluator 라우터

Evaluator의 평가 결과에 따라 **3가지 경로**로 분기합니다:

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    """Evaluator의 평가 결과에 따라 분기"""
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"      # 성공 또는 사용자 입력 필요 → 종료
    else:
        return "worker"   # 기준 미달 → 피드백과 함께 재작업

print("Evaluator 라우터 정의 완료")
print("  success_criteria_met=True  → END (작업 완료)")
print("  user_input_needed=True     → END (사용자 입력 대기)")
print("  둘 다 False               → worker (재작업)")

```
┌─────────────────────────────────────────────────────────────────────┐
│  04-2 vs 04-3 라우터 비교                                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  04-2 (tools_condition):                                           │
│    tool_calls 있음 → tools                                         │
│    tool_calls 없음 → END                                           │
│                                                                     │
│  04-3 (worker_router):                                             │
│    tool_calls 있음 → tools                                         │
│    tool_calls 없음 → evaluator  ← END 대신 evaluator!             │
│                                                                     │
│  04-3 (route_based_on_evaluation):                                 │
│    성공 or 질문     → END                                          │
│    기준 미달        → worker    ← 피드백과 함께 재작업             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 8. 그래프 구성 및 시각화

In [ ]:
# 그래프 구성

graph_builder = StateGraph(State)

# 노드 추가
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

# Edge 연결
graph_builder.add_edge(START, "worker")
graph_builder.add_conditional_edges(
    "worker", worker_router,
    {"tools": "tools", "evaluator": "evaluator"}
)
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges(
    "evaluator", route_based_on_evaluation,
    {"worker": "worker", "END": END}
)

# Compile
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("그래프 컴파일 완료")

In [ ]:
# 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))

### 그래프 구조 해석

시각화된 그래프에서 두 개의 루프를 확인할 수 있습니다:

1. **도구 사용 루프**: `worker → tools → worker` (Worker가 도구를 쓸 때)
2. **품질 검증 루프**: `worker → evaluator → worker` (Evaluator가 거부할 때)

```
add_conditional_edges의 세 번째 인자 — 라우팅 맵:

  worker_router가 반환하는 값     →  실제 이동할 노드
  ─────────────────────────────────────────────────
  "tools"                         →  "tools" 노드
  "evaluator"                     →  "evaluator" 노드

  route_based_on_evaluation이 반환하는 값  →  실제 이동할 노드
  ─────────────────────────────────────────────────
  "worker"                        →  "worker" 노드
  "END"                           →  END (그래프 종료)
```

---

## 9. 실행 테스트

In [ ]:
# 테스트: Playwright 브라우저 도구를 사용한 작업 (비동기)

config = {"configurable": {"thread_id": "test_1"}}

initial_state = {
    "messages": "https://www.example.com 에 접속해서 페이지 내용을 한국어로 요약해줘",
    "success_criteria": "example.com 페이지의 내용을 한국어로 3줄 이내로 요약할 것",
    "feedback_on_work": None,
    "success_criteria_met": False,
    "user_input_needed": False
}

result = await graph.ainvoke(initial_state, config=config)

print("=== 실행 결과 ===")
print(f"성공 기준 충족: {result['success_criteria_met']}")
print(f"사용자 입력 필요: {result['user_input_needed']}")
print()
print("=== 메시지 흐름 ===")
for msg in result["messages"]:
    role = msg.type if hasattr(msg, 'type') else msg.get('role', '?')
    content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
    tool_calls = getattr(msg, 'tool_calls', [])
    
    if tool_calls:
        for tc in tool_calls:
            print(f"  [{role}] tool_call → {tc['name']}({tc['args']})")
    elif content:
        print(f"  [{role}] {content[:120]}")

### 실행 흐름 분석

```
1. START → worker
   - 사용자 메시지 + 성공 기준을 받음
   - 필요하면 Playwright 브라우저 도구로 웹사이트 방문 (tool_call)

2. worker → tools
   - navigate_browser, extract_text 등 실행
   - 결과를 State에 추가

3. tools → worker
   - 도구 실행 결과를 바탕으로 응답 작성
   - tool_calls 없이 최종 응답 반환

4. worker → evaluator
   - Worker의 응답이 성공 기준을 충족하는지 평가
   - EvaluatorOutput으로 결과 반환

5. evaluator → END 또는 worker
   - 성공이면 END
   - 실패면 피드백과 함께 worker로 되돌림
```

---

## 10. Gradio UI — Sidekick 챗봇

이 패턴을 활용한 **Sidekick(개인 비서)** UI를 만듭니다. 사용자가 **작업 요청**과 **성공 기준**을 별도로 입력할 수 있습니다.

Worker가 Playwright 브라우저 도구를 사용하므로, 웹사이트 방문/텍스트 추출 등의 작업을 요청할 수 있습니다.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

def make_thread_id() -> str:
    return str(uuid.uuid4())


async def process_message(message, success_criteria, history, thread):
    """사용자 메시지를 처리하고 결과를 반환"""
    config = {"configurable": {"thread_id": thread}}

    state = {
        "messages": message,
        "success_criteria": success_criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False
    }
    result = await graph.ainvoke(state, config=config)
    
    # Worker의 응답과 Evaluator의 피드백을 분리하여 표시
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]


async def reset():
    """대화 초기화"""
    return "", "", None, make_thread_id()

print("Gradio 콜백 함수 정의 완료")

In [ ]:
# Sidekick UI

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Sidekick — 멀티 에이전트 개인 비서")
    gr.Markdown("작업을 요청하고 성공 기준을 설정하면, Worker가 Playwright 브라우저로 작업을 수행하고 Evaluator가 품질을 검증합니다.")
    thread = gr.State(make_thread_id())
    
    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=400, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(
                show_label=False,
                placeholder="작업 요청 (예: CNN 뉴스 헤드라인을 요약해줘)"
            )
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False,
                placeholder="성공 기준 (예: 한국어로 5개 헤드라인을 각 1줄로 요약)"
            )
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")
    
    # 이벤트 바인딩
    message.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    success_criteria.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread])

demo.launch()

### Gradio UI 구조

```
┌─────────────────────────────────────────────────────────────────────┐
│  Sidekick — 멀티 에이전트 개인 비서                                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌───────────────────────────────────────────────────────┐         │
│  │  Chatbot 영역                                        │         │
│  │  [user] CNN 뉴스 헤드라인 요약해줘                   │         │
│  │  [assistant] Worker의 응답 ...                       │         │
│  │  [assistant] Evaluator Feedback: 성공 기준 충족 ...  │         │
│  └───────────────────────────────────────────────────────┘         │
│                                                                     │
│  ┌───────────────────────────────────────────────────────┐         │
│  │  작업 요청: [입력 필드]                               │         │
│  │  성공 기준: [입력 필드]                               │         │
│  └───────────────────────────────────────────────────────┘         │
│                                                                     │
│  [ Reset ]  [ Go! ]                                                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**일반 ChatInterface와의 차이**: 입력 필드가 2개(작업 요청 + 성공 기준)이므로 `gr.Blocks`로 커스텀 UI를 구성합니다.

**Playwright 활용 예시**:
- 작업 요청: "CNN 뉴스 헤드라인을 요약해줘" → Worker가 실제로 CNN 방문 + 텍스트 추출 + 요약
- 성공 기준: "한국어로 5개 헤드라인을 각 1줄로 요약" → Evaluator가 기준 충족 여부 평가

---

## 정리

### Structured Output

```python
# Pydantic 모델 정의
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="...")
    success_criteria_met: bool = Field(description="...")

# LLM에 적용
llm_structured = llm.with_structured_output(EvaluatorOutput)
result = llm_structured.invoke("...")   # → EvaluatorOutput 객체
result.success_criteria_met              # → True/False (바로 사용)
```

### Worker-Evaluator 패턴

```
┌─────────────────────────────────────────────────────────────────────┐
│                 멀티 에이전트 핵심 요약                              │
├──────────────────────────────────┬──────────────────────────────────┤
│     구성요소                     │     역할                         │
├──────────────────────────────────┼──────────────────────────────────┤
│  Worker (LLM 1)                 │  도구로 작업 수행                │
│  Evaluator (LLM 2)              │  성공 기준 평가 + 피드백 생성   │
│  worker_router                  │  tool_calls 유무로 분기          │
│  route_based_on_evaluation      │  평가 결과로 분기                │
│  MemorySaver                    │  대화 기록 유지                  │
│  EvaluatorOutput                │  Structured Output 스키마       │
└──────────────────────────────────┴──────────────────────────────────┘
```

### 노트북 시리즈 전체 비교

| | 04-1 | 04-2 | 04-3 |
|---|---|---|---|
| **주제** | 기본 개념 | Tool Use | 멀티 에이전트 |
| **노드** | 1개 | 2개 | 3개 |
| **LLM** | 1개 | 1개 (bind_tools) | 2개 (bind_tools + structured) |
| **루프** | 없음 | 1개 (tool) | 2개 (tool + eval) |
| **State** | messages | messages | 복합 (5개 필드) |
| **새 개념** | State, Node, Edge | ToolNode, tools_condition | Structured Output, 멀티 에이전트 |

### 핵심 포인트

- **Structured Output**: `with_structured_output`으로 LLM 출력을 Pydantic 객체로 강제 → 프로그래밍으로 활용 가능
- **Worker-Evaluator**: 작업자 + 검수자 분리로 **자동 품질 검증 루프** 구현
- **복합 State**: messages 외에 여러 필드를 두어 그래프 전체에서 정보 공유
- **커스텀 라우터**: `add_conditional_edges`에 직접 작성한 라우터 함수 사용